In [ ]:
import sys
print("Python:", sys.version)
# Colab normally includes these packages. Install only if your environment lacks them.
# %pip install torch numpy scipy matplotlib
import os, random, urllib.request
import numpy as np
import scipy.sparse as sp
import torch
from collections import defaultdict
from time import time

def set_seed(seed=2021):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Enable GPU before running the training cell.")


Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PyTorch: 2.11.0+cpu | Device: cpu
Enable GPU before running the training cell.


## ตั้งค่าการทดลอง
`USE_DRIVE=True` จะขอเชื่อม Google Drive เพื่อเก็บผลข้าม session
`RUN_NAME` ควรเปลี่ยนเมื่อทดลองคนละ configuration เพื่อไม่เขียนทับผลเดิม
การรัน training cell ใหม่เริ่มฝึกใหม่ ไม่ใช่ resume; best_model.pt เก็บโมเดลที่ดีที่สุดสำหรับ inference
ค่าหลัก: 64 มิติ, 3 layers, ED 0.1, SSL weight 0.1, temperature 0.2, Adam 0.001
หาก GPU memory ไม่พอ ลด batch_size แล้วรายงานค่าที่เปลี่ยน อย่าอ้างว่าเทียบผล paper ได้ตรง ๆ

In [ ]:
from pathlib import Path
QUICK_RUN = False
USE_DRIVE = True
RUN_NAME = "sgl_yelp_quick" if QUICK_RUN else "sgl_yelp_20epochs2"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/SGL_team_results2")
else:
    OUTPUT_ROOT = Path("sgl_results")

cfg = dict(
    data_dir="./dataset/yelp2018", dataset="yelp2018", seed=2021,
    aug_type="ED", embed_size=64, n_layers=3,
    reg=1e-4, ssl_reg=0.1, ssl_ratio=0.1, ssl_temp=0.2,
    ssl_mode="both_side", lr=0.001, batch_size=2048,
    epochs=1 if QUICK_RUN else 20, stop_cnt=10,
    top_k=20, test_batch_size=128, progress_every=50,
    validation_fraction=0, require_gpu=True,
    output_dir=str(OUTPUT_ROOT / RUN_NAME),
)
print(cfg)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'data_dir': './dataset/yelp2018', 'dataset': 'yelp2018', 'seed': 2021, 'aug_type': 'ED', 'embed_size': 64, 'n_layers': 3, 'reg': 0.0001, 'ssl_reg': 0.1, 'ssl_ratio': 0.1, 'ssl_temp': 0.2, 'ssl_mode': 'both_side', 'lr': 0.001, 'batch_size': 2048, 'epochs': 20, 'stop_cnt': 10, 'top_k': 20, 'test_batch_size': 128, 'progress_every': 50, 'validation_fraction': 0, 'require_gpu': True, 'output_dir': '/content/drive/MyDrive/SGL_team_results2/sgl_yelp_20epochs2'}


## ข้อมูลและการสุ่มตัวอย่าง
ดาวน์โหลด Yelp2018 จาก SGL-Torch ครั้งแรก ถ้าดาวน์โหลดไม่ได้จะหยุด ไม่ใช้ข้อมูลจำลองแทนโดยไม่แจ้ง
Sampler สุ่ม user ก่อน แล้วสุ่ม positive/negative ตาม pipeline ของเพื่อน

In [ ]:
from __future__ import annotations
# ==============================================================================
# 2. Data Loading, Preprocessing, Graph Augmentation & Pairwise BPR Sampler
# ==============================================================================

def download_sgl_dataset(data_dir: str, dataset_name: str) -> bool:
    """
    Downloads official benchmark dataset files (<dataset>.train and <dataset>.test)
    directly from https://github.com/wujcan/SGL-Torch if requested in Google Colab.
    Supported official datasets: 'yelp2018', 'amazon-book', 'ifashion'.
    """
    if dataset_name not in ["yelp2018", "amazon-book", "ifashion"]:
        return False

    os.makedirs(data_dir, exist_ok=True)
    base_url = f"https://raw.githubusercontent.com/wujcan/SGL-Torch/master/dataset/{dataset_name}"
    for ext in ["train", "test"]:
        target_path = os.path.join(data_dir, f"{dataset_name}.{ext}")
        if not os.path.exists(target_path):
            url = f"{base_url}/{dataset_name}.{ext}"
            print(f"[Download] Fetching {url} ...")
            try:
                urllib.request.urlretrieve(url, target_path)
            except Exception as e:
                print(f"[Download] Could not fetch {url}: {e}")
                return False
    return True


def generate_synthetic_dataset(
    data_dir: str,
    dataset_name: str = "synthetic",
    num_users: int = 300,
    num_items: int = 500,
    interactions_per_user: int = 25,
    seed: int = 2021,
) -> None:
    """
    Generates a synthetic implicit-feedback dataset with a power-law item popularity
    distribution (reflecting the skewed long-tail distribution described in Section 1)
    and saves `<dataset_name>.train` and `<dataset_name>.test` in SGL-Torch CSV format.
    """
    os.makedirs(data_dir, exist_ok=True)
    rng = np.random.default_rng(seed)

    # Power-law item popularity distribution: few head items, many long-tail items
    item_ranks = np.arange(1, num_items + 1, dtype=np.float64)
    item_probs = item_ranks ** (-0.8)
    item_probs /= item_probs.sum()

    train_pairs, test_pairs = [], []

    for u in range(num_users):
        n_inter = max(5, int(rng.poisson(interactions_per_user)))
        n_inter = min(n_inter, num_items // 2)
        items = rng.choice(num_items, size=n_inter, replace=False, p=item_probs)

        # Split 80% train, 20% test
        n_test = max(1, int(len(items) * 0.2))
        train_items = np.sort(items[:-n_test])
        test_items = np.sort(items[-n_test:])

        for i in train_items:
            train_pairs.append(f"{u},{int(i)}")
        for i in test_items:
            test_pairs.append(f"{u},{int(i)}")

    with open(os.path.join(data_dir, f"{dataset_name}.train"), "w", encoding="utf-8") as f:
        f.write("\n".join(train_pairs) + "\n")

    with open(os.path.join(data_dir, f"{dataset_name}.test"), "w", encoding="utf-8") as f:
        f.write("\n".join(test_pairs) + "\n")


def sp_mat_to_sp_tensor(sp_mat: sp.spmatrix) -> torch.Tensor:
    """
    Converts a SciPy sparse matrix into a coalesced PyTorch sparse COO tensor.
    Directly mirrors `sp_mat_to_sp_tensor` in `SGL-Torch/util/pytorch/func.py`.
    """
    coo = sp_mat.tocoo().astype(np.float32)
    indices = torch.from_numpy(np.asarray([coo.row, coo.col], dtype=np.int64))
    values = torch.from_numpy(coo.data)
    return torch.sparse_coo_tensor(indices, values, coo.shape).coalesce()


class SGLDataset:
    """
    Dataset loader and graph builder for SGL, combining the functionality of
    `data/dataset.py` and `SGL.create_adj_mat` from `SGL-Torch`.
    """

    def __init__(
        self,
        data_dir: str = "./dataset/synthetic",
        dataset_name: str = "synthetic",
        ssl_ratio: float = 0.1,
        device: torch.device = DEVICE,
    ):
        self.data_dir = data_dir
        self.dataset_name = dataset_name
        self.ssl_ratio = ssl_ratio
        self.device = device

        train_path, test_path = self._resolve_data_paths()

        self.train_user_dict = defaultdict(list)
        self.test_user_dict = defaultdict(list)
        self.train_user_set = defaultdict(set)

        self.train_users = []
        self.train_items = []

        self.num_users = 0
        self.num_items = 0

        # Parse training and testing interactions
        self._load_file(train_path, is_train=True)
        self._load_file(test_path, is_train=False)

        self.train_users = np.asarray(self.train_users, dtype=np.int32)
        self.train_items = np.asarray(self.train_items, dtype=np.int32)
        self.num_ratings = len(self.train_users)

        self.train_user_dict = {
            u: np.asarray(items, dtype=np.int32) for u, items in self.train_user_dict.items()
        }
        self.test_user_dict = {
            u: np.asarray(items, dtype=np.int32) for u, items in self.test_user_dict.items()
        }

        # Build the intact normalized adjacency matrix D^{-1/2} A D^{-1/2} for the main task
        adj_matrix = self.create_adj_mat(is_subgraph=False)
        self.norm_adj = sp_mat_to_sp_tensor(adj_matrix).to(self.device)

    def _resolve_data_paths(self) -> tuple[str, str]:
        """Checks for existing `.train`/`.test` or `train.txt`/`test.txt`, downloads, or generates synthetic data."""
        sgl_train = os.path.join(self.data_dir, f"{self.dataset_name}.train")
        sgl_test = os.path.join(self.data_dir, f"{self.dataset_name}.test")
        txt_train = os.path.join(self.data_dir, "train.txt")
        txt_test = os.path.join(self.data_dir, "test.txt")

        if os.path.exists(sgl_train) and os.path.exists(sgl_test):
            return sgl_train, sgl_test
        if os.path.exists(txt_train) and os.path.exists(txt_test):
            return txt_train, txt_test

        # Attempt to download from official SGL-Torch GitHub if dataset_name is an official benchmark
        if download_sgl_dataset(self.data_dir, self.dataset_name):
            return sgl_train, sgl_test

        if self.dataset_name != "synthetic":
            raise RuntimeError("Yelp download failed. Stop here; do not substitute synthetic data.")
        print(f"[Preprocess] Generating synthetic dataset at '{self.data_dir}'...")
        generate_synthetic_dataset(self.data_dir, self.dataset_name)
        return sgl_train, sgl_test

    def _load_file(self, file_path: str, is_train: bool) -> None:
        """
        Reads interaction data supporting both:
        1. SGL-Torch CSV format (`user_id,item_id` per line)
        2. Adjacency list / whitespace-separated format (`user_id item_1 item_2 ...`)
        """
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split(",") if "," in line else line.split()
                if len(parts) < 2:
                    continue

                u = int(parts[0])
                items = [int(x) for x in parts[1:]]

                self.num_users = max(self.num_users, u + 1)
                self.num_items = max(self.num_items, max(items) + 1)

                for i in items:
                    if is_train:
                        if i not in self.train_user_set[u]:
                            self.train_user_dict[u].append(i)
                            self.train_user_set[u].add(i)
                            self.train_users.append(u)
                            self.train_items.append(i)
                    else:
                        self.test_user_dict[u].append(i)

    def create_adj_mat(self, is_subgraph: bool = False, aug_type: str = "ed") -> sp.spmatrix:
        """
        Builds and normalizes the bipartite user-item adjacency matrix.
        Directly mirrors `SGL.create_adj_mat` in `SGL-Torch/model/general_recommender/SGL.py`.
        """
        n_nodes = self.num_users + self.num_items
        users_np, items_np = self.train_users, self.train_items
        aug_type = aug_type.lower()

        if is_subgraph and self.ssl_ratio > 0:
            if aug_type == "nd":
                # Node Dropout (ND, Eq. 7): Drop `ssl_ratio` fraction of user and item nodes
                drop_user_size = int(self.num_users * self.ssl_ratio)
                drop_item_size = int(self.num_items * self.ssl_ratio)
                drop_user_idx = np.random.choice(self.num_users, size=drop_user_size, replace=False)
                drop_item_idx = np.random.choice(self.num_items, size=drop_item_size, replace=False)

                indicator_user = np.ones(self.num_users, dtype=np.float32)
                indicator_item = np.ones(self.num_items, dtype=np.float32)
                indicator_user[drop_user_idx] = 0.0
                indicator_item[drop_item_idx] = 0.0

                diag_indicator_user = sp.diags(indicator_user)
                diag_indicator_item = sp.diags(indicator_item)

                R = sp.csr_matrix(
                    (np.ones_like(users_np, dtype=np.float32), (users_np, items_np)),
                    shape=(self.num_users, self.num_items),
                )
                R_prime = diag_indicator_user.dot(R).dot(diag_indicator_item)
                user_np_keep, item_np_keep = R_prime.nonzero()
                ratings_keep = R_prime.data
                tmp_adj = sp.csr_matrix(
                    (ratings_keep, (user_np_keep, item_np_keep + self.num_users)),
                    shape=(n_nodes, n_nodes),
                )
            elif aug_type in ["ed", "rw"]:
                # Edge Dropout (ED, Eq. 8) & Random Walk per-layer (RW, Eq. 9):
                # Retain (1 - ssl_ratio) fraction of observed edges
                keep_size = int(len(users_np) * (1.0 - self.ssl_ratio))
                keep_idx = np.random.choice(len(users_np), size=keep_size, replace=False)
                user_np = np.array(users_np)[keep_idx]
                item_np = np.array(items_np)[keep_idx]
                ratings = np.ones_like(user_np, dtype=np.float32)
                tmp_adj = sp.csr_matrix(
                    (ratings, (user_np, item_np + self.num_users)),
                    shape=(n_nodes, n_nodes),
                )
            else:
                raise ValueError(f"Unsupported aug_type '{aug_type}'. Expected 'nd', 'ed', or 'rw'.")
        else:
            # Intact bipartite interaction graph G = (V, E)
            ratings = np.ones_like(users_np, dtype=np.float32)
            tmp_adj = sp.csr_matrix(
                (ratings, (users_np, items_np + self.num_users)),
                shape=(n_nodes, n_nodes),
            )

        # Symmetrize bipartite adjacency matrix: A = tmp_adj + tmp_adj^T
        adj_mat = tmp_adj + tmp_adj.T

        # Symmetric Laplacian normalization: D^{-1/2} A D^{-1/2}
        rowsum = np.array(adj_mat.sum(1))
        with np.errstate(divide="ignore"):
            d_inv = np.power(rowsum, -0.5).flatten()
        d_inv[np.isinf(d_inv)] = 0.0
        d_mat_inv = sp.diags(d_inv)
        norm_adj_tmp = d_mat_inv.dot(adj_mat)
        adj_matrix = norm_adj_tmp.dot(d_mat_inv)

        return adj_matrix

    def sample_subgraphs(
        self, aug_type: str = "ed", n_layers: int = 3
    ) -> tuple[torch.Tensor | list[torch.Tensor], torch.Tensor | list[torch.Tensor]]:
        """
        Generates two augmented graph views `sub_graph1` and `sub_graph2` per epoch:
        - For 'nd' or 'ed': returns two single sparse adjacency tensors shared across layers.
        - For 'rw': returns two lists of `n_layers` sparse adjacency tensors (one per layer).
        """
        aug_type = aug_type.lower()
        if aug_type in ["nd", "ed"]:
            sub_graph1 = sp_mat_to_sp_tensor(
                self.create_adj_mat(is_subgraph=True, aug_type=aug_type)
            ).to(self.device)
            sub_graph2 = sp_mat_to_sp_tensor(
                self.create_adj_mat(is_subgraph=True, aug_type=aug_type)
            ).to(self.device)
            return sub_graph1, sub_graph2
        elif aug_type == "rw":
            sub_graph1, sub_graph2 = [], []
            for _ in range(n_layers):
                tmp1 = self.create_adj_mat(is_subgraph=True, aug_type="rw")
                sub_graph1.append(sp_mat_to_sp_tensor(tmp1).to(self.device))
                tmp2 = self.create_adj_mat(is_subgraph=True, aug_type="rw")
                sub_graph2.append(sp_mat_to_sp_tensor(tmp2).to(self.device))
            return sub_graph1, sub_graph2
        else:
            raise ValueError(f"Invalid aug_type '{aug_type}'. Choose from ['nd', 'ed', 'rw'].")


class PairwiseSampler:
    """
    Pairwise BPR negative sampler matching `PairwiseSamplerV2` in `SGL-Torch/data/sampler.py`.
    Uniformly samples training triplets (u, i, j) where (u, i) in O^+ and (u, j) in O^-.
    """

    def __init__(self, dataset: SGLDataset, batch_size: int = 2048, shuffle: bool = True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.user_pos_dict = dataset.train_user_dict
        self.user_pos_set = dataset.train_user_set
        self.num_trainings = dataset.num_ratings
        self.num_items = dataset.num_items
        self.users = list(self.user_pos_dict.keys())

    def __iter__(self):
        sampled_users = np.random.choice(self.users, size=self.num_trainings, replace=True)
        unique_users, user_counts = np.unique(sampled_users, return_counts=True)

        all_users, all_pos_items, all_neg_items = [], [], []

        for u, count in zip(unique_users, user_counts):
            pos_items_u = self.user_pos_dict[u]
            pos_set_u = self.user_pos_set[u]

            pos_sampled = np.random.choice(pos_items_u, size=count, replace=True)
            neg_sampled = np.empty(count, dtype=np.int32)
            for idx in range(count):
                j = np.random.randint(0, self.num_items)
                while j in pos_set_u:
                    j = np.random.randint(0, self.num_items)
                neg_sampled[idx] = j

            all_users.append(np.full(count, u, dtype=np.int32))
            all_pos_items.append(pos_sampled)
            all_neg_items.append(neg_sampled)

        users_arr = np.concatenate(all_users)
        pos_arr = np.concatenate(all_pos_items)
        neg_arr = np.concatenate(all_neg_items)

        if self.shuffle:
            perm = np.random.permutation(self.num_trainings)
            users_arr = users_arr[perm]
            pos_arr = pos_arr[perm]
            neg_arr = neg_arr[perm]

        for start in range(0, self.num_trainings, self.batch_size):
            end = min(start + self.batch_size, self.num_trainings)
            yield users_arr[start:end], pos_arr[start:end], neg_arr[start:end]

## โมเดลของเรา
ใช้ SGLModel ที่เชื่อมกับ pipeline ของเพื่อนแล้ว; สมการ loss เป็น BPR + weighted InfoNCE + weighted L2

In [ ]:
from __future__ import annotations
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

class _SGLGraphEncoder(nn.Module):
    """Encoder API used by the team's eval branch main.py.

    Graphs and triplets are supplied by sgl.preprocess, not rebuilt here.
    norm_adj is a buffer so model.to(device) also moves the graph.
    """

    def __init__(self, num_users, num_items, norm_adj, embed_dim, n_layers, param_init):
        super().__init__()
        if tuple(norm_adj.shape) != (num_users + num_items, num_users + num_items):
            raise ValueError("norm_adj shape must match num_users + num_items")
        self.num_users, self.num_items = num_users, num_items
        self.n_layers = n_layers
        self.register_buffer("norm_adj", norm_adj, persistent=False)
        self.user_embeddings = nn.Embedding(num_users, embed_dim)
        self.item_embeddings = nn.Embedding(num_items, embed_dim)
        self._prediction_cache = None
        self.reset_parameters(init_method=param_init)

    def reset_parameters(self, pretrain=0, init_method="xavier_uniform", pretrain_dir=None):
        from pathlib import Path
        for name, embedding in (("user", self.user_embeddings), ("item", self.item_embeddings)):
            if pretrain:
                if pretrain_dir is None:
                    raise ValueError("pretrain_dir is required")
                values = torch.as_tensor(np.load(str(Path(pretrain_dir) / (name + "_embeddings.npy"))),
                                         dtype=embedding.weight.dtype, device=embedding.weight.device)
                if values.shape != embedding.weight.shape:
                    raise ValueError("Pretrained embedding shape does not match the ID mapping")
                with torch.no_grad():
                    embedding.weight.copy_(values)
            elif init_method == "xavier_uniform":
                nn.init.xavier_uniform_(embedding.weight)
            elif init_method == "xavier_normal":
                nn.init.xavier_normal_(embedding.weight)
            elif init_method == "normal":
                nn.init.normal_(embedding.weight, std=0.01)
            elif init_method == "uniform":
                nn.init.uniform_(embedding.weight, -0.01, 0.01)
            else:
                raise ValueError("Unsupported param_init: " + init_method)
        self._prediction_cache = None

    def train(self, mode=True):
        self._prediction_cache = None
        return super().train(mode)

    def _forward_gcn(self, graph):
        if isinstance(graph, (list, tuple)) and len(graph) != self.n_layers:
            raise ValueError("Layer-specific graphs must have n_layers entries")
        x = torch.cat((self.user_embeddings.weight, self.item_embeddings.weight))
        layers = [x]
        for k in range(self.n_layers):
            adjacency = graph[k] if isinstance(graph, (list, tuple)) else graph
            x = torch.sparse.mm(adjacency.to(x.device), x)
            layers.append(x)
        final = torch.stack(layers, dim=1).mean(dim=1)
        return final[:self.num_users], final[self.num_users:]

    @torch.no_grad()
    def predict(self, users):
        # Parameter versions change after optimizer steps and checkpoint loads.
        # This also prevents stale scores when evaluation follows a restore.
        key = (self.user_embeddings.weight._version, self.item_embeddings.weight._version,
               self.user_embeddings.weight.device, self.user_embeddings.weight.dtype)
        if self._prediction_cache is None or self._prediction_cache[0] != key:
            self._prediction_cache = (key, self._forward_gcn(self.norm_adj))
        user_repr, item_repr = self._prediction_cache[1]
        return user_repr[users] @ item_repr.t()


class SGLModel(nn.Module):
    """Drop-in model API for oransom48/SGL-implementation, branch eval.

    Reference interface inspected: sgl/main.py and sgl/evaluate.py, 2026-11-02.
    main.py owns sampling, optimizer, evaluation and stopping. calculate_loss
    returns (total, BPR, weighted_SSL, weighted_L2), all scalar tensors.
    predict returns NumPy scores [batch_users, num_items] in shared item-ID order.
    ED/ND take one adjacency per view; RW takes n_layers adjacencies per view.
    """

    def __init__(self, num_users, num_items, norm_adj, embed_dim=64, n_layers=3,
                 ssl_temp=0.2, ssl_reg=0.1, reg=1e-4, ssl_mode="both_side",
                 param_init="xavier_uniform"):
        super().__init__()
        if num_users <= 0 or num_items <= 0 or embed_dim <= 0 or n_layers < 0:
            raise ValueError("Invalid model dimensions")
        if ssl_temp <= 0 or ssl_reg < 0 or reg < 0:
            raise ValueError("Invalid loss hyperparameters")
        if ssl_mode not in ("both_side", "user_side", "item_side"):
            raise ValueError("Unsupported ssl_mode")
        self.num_users, self.num_items = num_users, num_items
        self.ssl_temp, self.ssl_reg, self.reg = ssl_temp, ssl_reg, reg
        self.ssl_mode = ssl_mode
        self.lightgcn = _SGLGraphEncoder(num_users, num_items, norm_adj,
                                         embed_dim, n_layers, param_init)

    def calculate_loss(self, sub_graph1, sub_graph2, bat_users, bat_pos_items, bat_neg_items):
        encoder = self.lightgcn
        device = encoder.user_embeddings.weight.device
        users, positives, negatives = [torch.as_tensor(x, dtype=torch.long, device=device)
                                      for x in (bat_users, bat_pos_items, bat_neg_items)]
        encoder._prediction_cache = None
        u, i = encoder._forward_gcn(encoder.norm_adj)
        margin = (u[users] * (i[positives] - i[negatives])).sum(dim=1)
        bpr = -F.logsigmoid(margin).sum()
        raw_reg = 0.5 * sum(x.pow(2).sum() for x in (
            encoder.user_embeddings(users), encoder.item_embeddings(positives),
            encoder.item_embeddings(negatives)))
        a_u, a_i = encoder._forward_gcn(sub_graph1)
        b_u, b_i = encoder._forward_gcn(sub_graph2)

        def contrastive(a, b, ids):
            a, b = F.normalize(a, dim=1), F.normalize(b, dim=1)
            anchors = a[ids]
            positive = (anchors * b[ids]).sum(dim=1, keepdim=True)
            return torch.logsumexp((anchors @ b.t() - positive) / self.ssl_temp, dim=1).sum()

        ssl = bpr.new_zeros(())
        if self.ssl_mode in ("user_side", "both_side"):
            ssl = ssl + contrastive(a_u, b_u, users)
        if self.ssl_mode in ("item_side", "both_side"):
            ssl = ssl + contrastive(a_i, b_i, positives)
        weighted_ssl, weighted_reg = self.ssl_reg * ssl, self.reg * raw_reg
        return bpr + weighted_ssl + weighted_reg, bpr, weighted_ssl, weighted_reg

    @torch.no_grad()
    def predict(self, users):
        users = torch.as_tensor(users, dtype=torch.long,
                                device=self.lightgcn.user_embeddings.weight.device)
        if users.ndim != 1:
            raise ValueError("users must be a one-dimensional sequence")
        return self.lightgcn.predict(users).cpu().numpy()


__all__ = ["SGLRecommender", "SGLModel"]



## Evaluation ของเพื่อน
โค้ดการจัดอันดับและคำนวณ Precision, Recall, NDCG และกลุ่มความนิยมมาจาก notebook ของเพื่อน
mask รายการที่เคยเห็นก่อนเลือก Top-20

In [ ]:
from __future__ import annotations
# ==============================================================================
# 4. Top-K All-Ranking Evaluator & Long-Tail Group Decomposition
# ==============================================================================

def compute_user_metrics(
    ranked_items: np.ndarray,
    ground_truth: set[int],
    top_k_list: list[int],
) -> dict[str, np.ndarray]:
    """
    Computes Precision@K, Recall@K, and NDCG@K for a single user u.
    Exact port of `precision`, `recall`, and `ndcg` in `SGL-Torch/reckit/cython/include/metric.h`.
    """
    max_k = len(ranked_items)
    truth_len = len(ground_truth)

    hits = np.array(
        [1.0 if item in ground_truth else 0.0 for item in ranked_items], dtype=np.float64
    )
    cum_hits = np.cumsum(hits)
    ranks = np.arange(1, max_k + 1, dtype=np.float64)

    precisions = cum_hits / ranks
    recalls = cum_hits / truth_len if truth_len > 0 else np.zeros_like(cum_hits)

    discounts = 1.0 / np.log2(np.arange(2, max_k + 2, dtype=np.float64))
    dcg = np.cumsum(hits * discounts)

    ideal_hits = np.zeros(max_k, dtype=np.float64)
    ideal_hits[: min(max_k, truth_len)] = 1.0
    idcg = np.cumsum(ideal_hits * discounts)
    ndcgs = np.where(idcg > 0, dcg / idcg, 0.0)

    k_indices = [k - 1 for k in top_k_list]
    return {
        "Precision": precisions[k_indices],
        "Recall": recalls[k_indices],
        "NDCG": ndcgs[k_indices],
    }


class Evaluator:
    """All-ranking Top-K Evaluator matching `Evaluator` in `SGL-Torch/reckit/evaluator.py`."""

    def __init__(
        self,
        user_train_dict: dict[int, np.ndarray],
        user_test_dict: dict[int, np.ndarray],
        metrics: list[str] | None = None,
        top_k: int | list[int] = 20,
        batch_size: int = 128,
    ):
        if metrics is None:
            metrics = ["Precision", "Recall", "NDCG"]
        elif isinstance(metrics, str):
            metrics = [metrics]

        self.user_pos_train = user_train_dict if user_train_dict is not None else {}
        self.user_pos_test = user_test_dict
        self.metrics = metrics
        self.batch_size = batch_size
        self.top_show = [top_k] if isinstance(top_k, int) else sorted(list(top_k))
        self.max_top = max(self.top_show)

    def metrics_info(self) -> str:
        headers = []
        for k in self.top_show:
            for m in self.metrics:
                headers.append(f"{m}@{k}".ljust(12))
        return "\t".join(headers)

    def evaluate(self, model, test_users: list[int] | None = None) -> tuple[np.ndarray, str]:
        model.eval()
        if test_users is None:
            test_users = [u for u, items in self.user_pos_test.items() if len(items) > 0]

        all_user_results = []

        for start in range(0, len(test_users), self.batch_size):
            batch_users = test_users[start : start + self.batch_size]
            ranking_score = model.predict(batch_users)
            ranking_score = np.array(ranking_score, dtype=np.float32, copy=True)

            # Mask observed training items with -inf so they are never recommended
            for idx, u in enumerate(batch_users):
                if u in self.user_pos_train and len(self.user_pos_train[u]) > 0:
                    ranking_score[idx, self.user_pos_train[u]] = -np.inf

            # Fast Top-K extraction via argpartition
            top_k_part = np.argpartition(-ranking_score, self.max_top - 1, axis=1)[:, : self.max_top]
            row_indices = np.arange(len(batch_users))[:, None]
            top_k_scores = ranking_score[row_indices, top_k_part]
            top_k_order = np.argsort(-top_k_scores, axis=1)
            top_k_items = top_k_part[row_indices, top_k_order]

            for idx, u in enumerate(batch_users):
                truth_set = set(self.user_pos_test[u].tolist())
                user_metric_dict = compute_user_metrics(
                    ranked_items=top_k_items[idx],
                    ground_truth=truth_set,
                    top_k_list=self.top_show,
                )
                user_vec = []
                for k_idx in range(len(self.top_show)):
                    for m in self.metrics:
                        user_vec.append(user_metric_dict[m][k_idx])
                all_user_results.append(user_vec)

        all_user_results = np.asarray(all_user_results, dtype=np.float64)
        final_result = np.mean(all_user_results, axis=0)
        buf = "\t".join([("%.8f" % x).ljust(12) for x in final_result])
        return final_result, buf

    def evaluate_long_tail_groups(
        self,
        model,
        train_items: np.ndarray,
        num_items: int,
        num_groups: int = 10,
        k: int = 20,
    ) -> np.ndarray:
        """
        Decomposes Recall@K across 10 item popularity groups (Section 4.3.1, Figure 4).
        Group 1 = lowest-degree (long-tail) items; Group 10 = highest-degree (head) items.
        """
        model.eval()
        item_degrees = np.bincount(train_items, minlength=num_items)
        sorted_items = np.argsort(item_degrees)
        sorted_degrees = item_degrees[sorted_items]
        cum_degrees = np.cumsum(sorted_degrees)
        total_interactions = cum_degrees[-1]

        item_to_group = np.zeros(num_items, dtype=np.int32)
        thresholds = np.linspace(0, total_interactions, num_groups + 1)[1:]
        group_ids = np.clip(np.searchsorted(thresholds, cum_degrees, side="left"), 0, num_groups - 1)
        item_to_group[sorted_items] = group_ids

        test_users = [u for u, items in self.user_pos_test.items() if len(items) > 0]
        group_recalls = np.zeros(num_groups, dtype=np.float64)

        for start in range(0, len(test_users), self.batch_size):
            batch_users = test_users[start : start + self.batch_size]
            ranking_score = np.array(model.predict(batch_users), dtype=np.float32, copy=True)

            for idx, u in enumerate(batch_users):
                if u in self.user_pos_train and len(self.user_pos_train[u]) > 0:
                    ranking_score[idx, self.user_pos_train[u]] = -np.inf

            top_k_part = np.argpartition(-ranking_score, k - 1, axis=1)[:, :k]

            for idx, u in enumerate(batch_users):
                truth_set = set(self.user_pos_test[u].tolist())
                truth_len = len(truth_set)
                for rec_item in top_k_part[idx]:
                    if rec_item in truth_set:
                        g = item_to_group[int(rec_item)]
                        group_recalls[g] += 1.0 / truth_len

        group_recalls /= max(1, len(test_users))
        return group_recalls

## วงรอบฝึกและบันทึกผล
สร้างกราฟจาก train เท่านั้น; เลือก epoch ด้วย validation และประเมิน test หลังโหลด weights ที่ดีที่สุด
หาก validation_fraction=0 จะเลือกจาก test ตาม notebook เดิมของเพื่อน
บันทึก config, split, dataset hashes, loss/metrics และ best weights ทุก epoch

In [ ]:
import copy
import csv
import hashlib
import json
import platform
from pathlib import Path
from time import time

def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def run_experiment(cfg):
    set_seed(cfg["seed"])
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if cfg["require_gpu"] and device.type != "cuda":
        raise RuntimeError("Enable a GPU runtime in Colab, reconnect, and run again.")
    out = Path(cfg["output_dir"])
    out.mkdir(parents=True, exist_ok=True)
    save_json(out / "config.json", cfg)
    dataset = SGLDataset(data_dir=cfg["data_dir"], dataset_name=cfg["dataset"],
                         ssl_ratio=cfg["ssl_ratio"], device=device)
    original_train = {u: a.copy() for u, a in dataset.train_user_dict.items()}
    validation = {}
    fraction = cfg["validation_fraction"]
    if not 0 <= fraction < 1:
        raise ValueError("validation_fraction must be in [0, 1)")
    if fraction:
        rng = np.random.RandomState(cfg["seed"])
        reduced = {}
        for u, items in original_train.items():
            shuffled = rng.permutation(items)
            count = min(len(items) - 1, max(1, int(len(items) * fraction))) if len(items) >= 2 else 0
            reduced[u] = shuffled[count:].astype(np.int32)
            if count:
                validation[u] = shuffled[:count].astype(np.int32)
        if not validation:
            raise ValueError("Dataset is too small for a validation split")
        dataset.train_user_dict = reduced
        dataset.train_user_set = {u: set(a.tolist()) for u, a in reduced.items()}
        dataset.train_users = np.concatenate([np.full(len(a), u, dtype=np.int32) for u, a in reduced.items()])
        dataset.train_items = np.concatenate(list(reduced.values())).astype(np.int32)
        dataset.num_ratings = len(dataset.train_users)
        dataset.norm_adj = sp_mat_to_sp_tensor(dataset.create_adj_mat()).to(device)
        selection_truth = validation
        selection_name = "validation"
    else:
        selection_truth = dataset.test_user_dict
        selection_name = "test (friend protocol; test used to choose epoch)"
    # Save exact graph/split inputs, in addition to file hashes and seed.
    np.savez_compressed(out / "training_pairs.npz", users=dataset.train_users, items=dataset.train_items)
    save_json(out / "validation.json", {str(u): a.tolist() for u, a in validation.items()})
    candidates = [len(dataset.train_user_dict[u]) for u in selection_truth]
    if dataset.num_items - max(candidates, default=0) < cfg["top_k"]:
        raise ValueError("Fewer than top_k unmasked candidate items for some users")
    if any(len(items) >= dataset.num_items for items in dataset.train_user_dict.values()):
        raise ValueError("A user has no negative items available for BPR sampling")
    for u, truth in dataset.test_user_dict.items():
        if set(truth.tolist()).intersection(original_train.get(u, [])):
            raise ValueError("Training and test interactions overlap")
    if any(dataset.num_items - len(original_train.get(u, [])) < cfg["top_k"] for u in dataset.test_user_dict):
        raise ValueError("Test ranking has fewer than top_k unseen candidate items")
    evaluator = Evaluator(dataset.train_user_dict, selection_truth,
                          top_k=[cfg["top_k"]], batch_size=cfg["test_batch_size"])
    test_evaluator = Evaluator(original_train, dataset.test_user_dict,
                               top_k=[cfg["top_k"]], batch_size=cfg["test_batch_size"])
    sampler = PairwiseSampler(dataset, batch_size=cfg["batch_size"], shuffle=True)
    model = SGLModel(dataset.num_users, dataset.num_items, dataset.norm_adj,
                     embed_dim=cfg["embed_size"], n_layers=cfg["n_layers"],
                     ssl_temp=cfg["ssl_temp"], ssl_reg=cfg["ssl_reg"],
                     reg=cfg["reg"], ssl_mode=cfg["ssl_mode"]).to(device)
    optimizer = torch.optim.Adam(model.lightgcn.parameters(), lr=cfg["lr"])
    history, best_weights = [], None
    best_recall, best_epoch, stalled = -float("inf"), 0, 0
    print("Model:", type(model).__name__, "| device:", device)
    print("Users:", dataset.num_users, "Items:", dataset.num_items,
          "Train interactions:", dataset.num_ratings, "Selection:", selection_name)
    print("Selection metrics:", evaluator.metrics_info())
    started = time()
    for epoch in range(1, cfg["epochs"] + 1):
        epoch_start = time()
        model.train()
        view_a, view_b = dataset.sample_subgraphs(cfg["aug_type"].lower(), cfg["n_layers"])
        totals = np.zeros(4, dtype=np.float64)
        for step, (users, positives, negatives) in enumerate(sampler, 1):
            optimizer.zero_grad()
            parts = model.calculate_loss(view_a, view_b, users, positives, negatives)
            if not torch.isfinite(parts[0]):
                raise FloatingPointError("Non-finite training loss")
            parts[0].backward()
            optimizer.step()
            totals += np.asarray([part.detach().item() for part in parts])
            if cfg["progress_every"] and step % cfg["progress_every"] == 0:
                print("Epoch", epoch, "| batch", step, "| elapsed", round(time() - epoch_start), "s", flush=True)
        metrics, formatted = evaluator.evaluate(model)
        if not np.isfinite(metrics).all():
            raise FloatingPointError("Non-finite evaluation metrics")
        row = dict(epoch=epoch, total_loss=float(totals[0] / dataset.num_ratings),
                   bpr_loss=float(totals[1] / dataset.num_ratings),
                   ssl_loss=float(totals[2] / dataset.num_ratings),
                   reg_loss=float(totals[3] / dataset.num_ratings),
                   precision=float(metrics[0]), recall=float(metrics[1]), ndcg=float(metrics[2]),
                   seconds=time() - epoch_start)
        history.append(row)
        print("Epoch {epoch}: loss={total_loss:.4f} = {bpr_loss:.4f} + {ssl_loss:.4f} + {reg_loss:.4f}; {seconds:.1f}s".format(**row))
        print(selection_name, formatted)
        if row["recall"] > best_recall:
            best_recall, best_epoch, stalled = row["recall"], epoch, 0
            best_weights = {k: v.detach().cpu().clone() for k, v in model.lightgcn.state_dict().items()}
            torch.save({"weights": best_weights, "epoch": epoch, "config": cfg,
                        "num_users": dataset.num_users, "num_items": dataset.num_items},
                       str(out / "best_model.pt"))
        else:
            stalled += 1
        # Save each completed epoch so results survive an interrupted later epoch.
        with (out / "history.csv").open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(row))
            writer.writeheader()
            writer.writerows(history)
        save_json(out / "history.json", history)
        if stalled >= cfg["stop_cnt"]:
            print("Early stopping at epoch", epoch)
            break
    model.lightgcn.load_state_dict(best_weights)
    final_metrics, formatted = test_evaluator.evaluate(model)
    train_items_for_groups = np.concatenate(list(original_train.values()))
    groups = test_evaluator.evaluate_long_tail_groups(
        model, train_items_for_groups, dataset.num_items, num_groups=10, k=cfg["top_k"])
    result = dict(best_epoch=best_epoch, selection_split=selection_name,
                  selection_recall=best_recall,
                  test_precision=float(final_metrics[0]), test_recall=float(final_metrics[1]),
                  test_ndcg=float(final_metrics[2]), group_recalls=groups.tolist(),
                  epochs_completed=len(history), total_seconds=time() - started,
                  python=platform.python_version(), torch=torch.__version__,
                  numpy=np.__version__, device=str(device),
                  gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None,
                  num_users=dataset.num_users, num_items=dataset.num_items,
                  train_interactions=dataset.num_ratings)
    result["dataset_hashes"] = {
        ext: file_sha256(Path(cfg["data_dir"]) / (cfg["dataset"] + "." + ext))
        for ext in ("train", "test")
    }
    save_json(out / "results.json", result)
    print("\nBest epoch:", best_epoch, "| selected using:", selection_name)
    print("FINAL TEST:", test_evaluator.metrics_info())
    print(formatted)
    print("Long-tail groups:", groups)
    print("Saved to:", out.resolve())
    return model, history, result


In [ ]:
trained_model, training_history, results = run_experiment(cfg)

RuntimeError: Enable a GPU runtime in Colab, reconnect, and run again.

## กราฟและดาวน์โหลดผล
กราฟกลางเป็นคะแนนของ split ที่ใช้เลือก epoch ส่วน test สุดท้ายอยู่ใน results.json
หากตั้ง validation_fraction=0 กราฟกลางจะเป็น test เหมือนภาพของเพื่อน

In [ ]:
import matplotlib.pyplot as plt
epochs = [r["epoch"] for r in training_history]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(epochs, [r["bpr_loss"] for r in training_history], marker="o", label="BPR")
axes[0].plot(epochs, [r["ssl_loss"] for r in training_history], marker="s", label="Weighted SSL")
axes[0].set(title="Training losses", xlabel="Epoch", ylabel="Loss / sampled triplet")
axes[0].legend()
axes[1].plot(epochs, [r["recall"] for r in training_history], marker="o", label="Recall@20")
axes[1].plot(epochs, [r["ndcg"] for r in training_history], marker="^", label="NDCG@20")
axes[1].set(title="Selection: " + results["selection_split"], xlabel="Epoch", ylabel="Metric")
axes[1].legend()
axes[2].bar(np.arange(1, 11), results["group_recalls"])
axes[2].set(title="Final test: popularity-group Recall@20", xlabel="Group (tail to head)", ylabel="Recall contribution")
axes[2].set_xticks(np.arange(1, 11))
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(Path(cfg["output_dir"]) / "training_curves.png", dpi=160)
plt.show()

import shutil
archive_path = shutil.make_archive(str(Path(cfg["output_dir"]).resolve()) + "_results",
                                  "zip", cfg["output_dir"])
print("Results ZIP:", archive_path)
# Colab: uncomment the following two lines to download the ZIP.
# from google.colab import files
# files.download(archive_path)


In [ ]:
for name, values in trained_model.lightgcn.state_dict().items():
    print(name, tuple(values.shape))

print("ตัวเลข 10 ค่าแรกของ User 42:")
print(
    trained_model.lightgcn.user_embeddings.weight[42, :10]
    .detach().cpu().numpy()
)

In [ ]:
import torch

path = "/content/drive/MyDrive/SGL_team_results/sgl_yelp_20epochs/best_model.pt"

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=True,
)

weights = checkpoint["weights"]

print("โมเดลที่บันทึกไว้จาก epoch:", checkpoint["epoch"])

for name, values in weights.items():
    print(name, tuple(values.shape))

print("\nตัวเลข 10 ค่าแรกของ User 42:")
print(weights["user_embeddings.weight"][42, :10].numpy())

## ขอบเขตการตรวจ
เตรียมให้ใช้กับ Colab GPU แต่ยังไม่มีการฝึก Yelp2018 เต็มชุดบน GPU ในการสร้างไฟล์นี้
ทดสอบ pipeline ด้วยข้อมูลจำลองเล็กบน CPU เท่านั้น ผลจากข้อมูลจำลองไม่ใช้รายงานผลวิจัย
การเปลี่ยนจำนวน epoch, split, batch size หรือ embedding size ต้องระบุใน presentation